# L3 — Screw motion
**Foundations of Modern Robotics · Majid Khadiv · ATARI Lab, TUM**

This notebook produces the course's interactive replacement for the external screw-axis visualizer linked from **L3, slide 32: “The screw interpretation of a twist.”** It also includes the worked example from slide 33.

**Run all cells.** The last cell displays the GUI and writes `L3_screw_motion.html` next to this notebook. Open that HTML file directly for a larger classroom view. The generated file includes its plotting library: its controls and animation work offline, without a running Python kernel. GitHub's notebook preview itself does not execute the GUI.

Install the dependencies in your notebook environment once:
```bash
python -m pip install numpy scipy plotly jupyterlab
```
If a notebook viewer suppresses interactive HTML, open the generated HTML file in a browser. In Jupyter, execute the cells yourself to produce trusted output.

## What to try

1. **Off-axis helix:** play the motion and observe the frame turning while its origin follows a curved path. Orange and teal velocity arrows add to the dark arrow.
2. **Pure rotation:** the pitch is zero; a nonzero linear twist component can still arise from an offset axis.
3. **Slide 33:** recover $u=(0,1,1)/\sqrt2$, $r_0=(0,1,-1)$ m, and $h=0$ from $\mathcal V_s=(4,0,0,\,0,2,2)$. The duration is $0.5$ s.
4. **Pure translation:** rotation vanishes, so there is no unique finite screw axis or finite pitch.
5. **180° rotation:** the logarithm needs a branch choice; the rotation itself remains regular.
6. Choose **Screw parameters** and enter $450^\circ$. This preserves the extra turn. Switching to pose input and applying the same endpoint recovers a principal rotation, which cannot infer winding history.


## Geometry, pose increments, and physical velocity

We use $T=T_{sb}$, mapping body coordinates into the world frame. The initial pose is $T_0=I$. All displayed points and axes use world coordinates, and all frames are right-handed.

For a target pose $T_1$, first compute a pose increment $\xi=(\rho,\phi)=\operatorname{Log}(T_1)$ and interpolate:
\[
T(s)=\exp(s\xi^\wedge),\quad 0\le s\le1,\qquad
\xi^\wedge=\begin{bmatrix}\phi^\wedge&\rho\\0&0\end{bmatrix}.
\]
For a motion duration $\Delta t$ with $s=t/\Delta t$, the constant **spatial twist** is
\[
\mathcal V_s=(v,\omega)=\xi/\Delta t,
\qquad \dot T=\mathcal V_s^\wedge T.
\]
This distinction keeps a pose increment separate from a physical velocity. Twists are **translation-first**, matching the course and Pinocchio.

For $\omega\ne0$:
\[
u=\frac{\omega}{\|\omega\|},\qquad
h=\frac{\omega^Tv}{\|\omega\|^2},\qquad
r_0=\frac{\omega\times v}{\|\omega\|^2},\qquad
\dot x=\underbrace{\omega\times(x-r_0)}_{\text{rotation}}+
\underbrace{h\omega}_{\text{translation}}.
\]
The axis is $r(\lambda)=r_0+\lambda u$, with $u^Tr_0=0$. The pitch $h$ is in metres per radian; axial travel after rotation $\theta$ is $h\theta$.

The GUI marks a fixed material point $x_b=(0.32,0.22,0.17)$ m on the body. Its world position is $x=Rx_b+p$. The body-origin velocity is $\dot p=v+\omega\times p$, whereas the body twist is $\mathcal V_b=(R^T\dot p,R^T\omega)$. The matrix $\dot T$ is neither of these six-vectors. For this particular path, $T_0=I$ and the constant generator commutes with $T(s)$, so $\mathcal V_b=\mathcal V_s$. This numerical equality does not hold for arbitrary rigid-body motions.

Pose inputs use the principal rotation branch $0\le\|\phi\|\le\pi$. At $\pi$, the branch is not unique. Direct screw inputs preserve the specified winding. Pure translation and identity are handled separately without dividing by $\|\omega\|^2$.


In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation


def hat3(a):
    """a^wedge b = a x b, with a expressed in a right-handed frame."""
    x, y, z = np.asarray(a, dtype=float)
    return np.array([[0., -z, y], [z, 0., -x], [-y, x, 0.]])


def hat6(V):
    """Course convention: V = (v, omega), linear part first."""
    V = np.asarray(V, dtype=float)
    A = np.zeros((4, 4))
    A[:3, :3] = hat3(V[3:])
    A[:3, 3] = V[:3]
    return A


def exp_se3(V, t=1.):
    """exp(t V^wedge), including pure translation and small rotations."""
    V = np.asarray(V, dtype=float)
    rho, phi = t * V[:3], t * V[3:]
    theta = np.linalg.norm(phi)
    A = hat3(phi)
    if theta < 1e-4:
        q = theta * theta
        a = 1 - q / 6 + q*q / 120
        b = .5 - q / 24 + q*q / 720
        c = 1/6 - q / 120 + q*q / 5040
    else:
        a = np.sin(theta) / theta
        b = 2 * np.sin(theta / 2)**2 / theta**2
        c = (theta - np.sin(theta)) / theta**3
    T = np.eye(4)
    T[:3, :3] = np.eye(3) + a * A + b * A @ A
    T[:3, 3] = (np.eye(3) + b * A + c * A @ A) @ rho
    return T


def screw_parameters(V):
    """Return unit direction, pitch, and closest axis point; no axis for omega=0."""
    v, omega = np.asarray(V[:3], float), np.asarray(V[3:], float)
    speed = np.linalg.norm(omega)
    if speed == 0:
        return None, None, None
    return omega / speed, omega @ v / speed**2, np.cross(omega, v) / speed**2


def point_velocity(V, x):
    return np.asarray(V[:3]) + np.cross(V[3:], x)


def body_twist(V_s, T_sb):
    R, p = T_sb[:3, :3], T_sb[:3, 3]
    return np.r_[R.T @ point_velocity(V_s, p), R.T @ V_s[3:]]


def log_se3(T):
    """Principal rotation branch: phi has norm <= pi; return (rho, phi)."""
    T = np.asarray(T, float)
    R, p = T[:3, :3], T[:3, 3]
    if not np.allclose(R.T @ R, np.eye(3), atol=1e-9) or not np.isclose(np.linalg.det(R), 1):
        raise ValueError('R must be a proper rotation matrix.')
    phi = Rotation.from_matrix(R).as_rotvec()
    theta = np.linalg.norm(phi)
    A = hat3(phi)
    if theta < 1e-4:
        q = theta**2
        b, c = .5-q/24+q*q/720, 1/6-q/120+q*q/5040
    else:
        b = 2*np.sin(theta/2)**2/theta**2
        c = (theta-np.sin(theta))/theta**3
    rho = np.linalg.solve(np.eye(3)+b*A+c*A@A, p)
    return np.r_[rho, phi]


## A numerical example and independent checks
The checks compare our exponential with `scipy.linalg.expm`, and verify the slide-33 screw axis and the point-velocity formula.

In [ ]:
from scipy.linalg import expm

V_example = np.array([4., 0., 0., 0., 2., 2.])
u, h, r0 = screw_parameters(V_example)
np.testing.assert_allclose(u, [0, 1/np.sqrt(2), 1/np.sqrt(2)])
np.testing.assert_allclose(r0, [0, 1, -1])
assert np.isclose(h, 0)

cases = [V_example, np.zeros(6), np.array([1., 2., 3., 0., 0., 0.]),
         np.array([.2, -.1, .5, 1e-9, -2e-9, 3e-9])]
for V in cases:
    for t in [-1., 0., .5, 2.]:
        T = exp_se3(V, t)
        np.testing.assert_allclose(T, expm(t * hat6(V)), atol=1e-10)
        np.testing.assert_allclose(T[:3, :3].T @ T[:3, :3], np.eye(3), atol=1e-10)
        assert np.isclose(np.linalg.det(T[:3, :3]), 1.)

x0 = np.array([.32, .22, .17, 1.])
t, eps = .2, 1e-6
x = (exp_se3(V_example, t) @ x0)[:3]
finite_difference = ((exp_se3(V_example, t+eps) @ x0 -
                      exp_se3(V_example, t-eps) @ x0) / (2*eps))[:3]
np.testing.assert_allclose(finite_difference, point_velocity(V_example, x), atol=1e-8)
print('Checks passed. Slide 33:')
print('u =', u, '\nr0 =', r0, '\nh =', h)


## Build the browser GUI

The following two cells contain the HTML/CSS layout and the JavaScript controller. They are collapsed for readability. The browser uses the same Rodrigues/left-Jacobian formulas as the Python functions above; its rotation-log implementation uses quaternion extraction to remain well behaved at a half-turn. Invalid rotation matrices are rejected instead of silently repaired.

The **pose** and **matrix** editors describe a target displacement, while the **screw** editor describes its generating motion. “Apply displacement” synchronizes the other editors. Frame axes and velocity arrows use separate legends and scales.


In [ ]:
HTML_TEMPLATE = r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>L3 · Screw motion — ATARI Lab</title>
<style>
*{box-sizing:border-box}body{margin:0;font:14px/1.45 system-ui,-apple-system,sans-serif;color:#24364B;background:#F4F7FB}
header{padding:17px 26px;background:#fff;border-bottom:3px solid #0065BD;display:flex;align-items:center;justify-content:space-between;gap:20px}
h1{font-size:24px;margin:0;color:#00589E}header p{margin:3px 0 0;color:#667789}.badge{font-size:12px;letter-spacing:.08em;color:#0065BD;font-weight:700}
main{max-width:1550px;margin:auto;padding:16px;display:grid;grid-template-columns:315px minmax(400px,1fr);gap:16px}
.card{background:#fff;border:1px solid #DCE4ED;border-radius:10px;padding:15px}.side{display:flex;flex-direction:column;gap:12px}
h2{font-size:16px;margin:0 0 10px}.label{font-size:12px;font-weight:650;display:block;margin:10px 0 4px;color:#43566D}
select,input[type=number],button{font:inherit;border:1px solid #CAD5E1;border-radius:5px;padding:7px;background:white;color:inherit}
select{width:100%}input[type=number]{width:100%;min-width:0}button{cursor:pointer}button:hover{background:#EDF4FC;border-color:#0065BD}
button.primary{background:#0065BD;color:white;border-color:#0065BD}button:disabled{opacity:.45;cursor:default}
.three{display:grid;grid-template-columns:repeat(3,1fr);gap:6px}.three label{font-size:11px;color:#607287}.matrix{display:grid;grid-template-columns:repeat(4,1fr);gap:4px}.matrix input{padding:5px;font-size:12px}
.hint{font-size:12px;color:#64758A;margin:7px 0}.input-panel[hidden]{display:none}.two{display:grid;grid-template-columns:1fr 1fr;gap:8px}.buttons{display:flex;gap:7px;margin-top:9px}.buttons button{flex:1}
input[type=range]{width:100%;accent-color:#0065BD}.progress{display:flex;justify-content:space-between;font-size:12px}.check{display:block;font-size:12px;margin-top:6px}
#status{font-size:12px;min-height:18px;color:#63778D}#status.error{color:#B3261E;font-weight:600}
#plot{width:100%;height:570px}.scene{padding:0;overflow:hidden}.legend{border-top:1px solid #E5EAF1;padding:11px 16px;display:flex;flex-wrap:wrap;gap:7px 17px;font-size:12px}
.legend i{display:inline-block;width:18px;height:3px;vertical-align:middle;margin-right:5px}.readouts{grid-column:1/-1;display:grid;grid-template-columns:1fr 1fr 1fr;gap:14px}.readouts h2{font-size:14px;color:#00589E}.readouts pre{font:12px/1.55 ui-monospace,SFMono-Regular,monospace;white-space:pre-wrap;margin:6px 0}
.equation{font-size:16px;background:#EDF5FC;border-radius:5px;padding:10px 12px;margin-bottom:10px}.small{font-size:12px;color:#63758A}footer{max-width:1500px;padding:0 16px 18px;margin:auto;font-size:11px;color:#64758A}a{color:#0065BD}details{margin-top:8px}summary{cursor:pointer}
@media(max-width:850px){main{display:flex;flex-direction:column}.readouts{display:flex;flex-direction:column}.side{display:grid;grid-template-columns:1fr 1fr}.scene{min-width:0}#plot{height:450px}.badge{display:none}}
@media(max-width:550px){.side{display:flex}header{padding:12px 16px}h1{font-size:21px}}
</style><script>__PLOTLY__</script></head><body>
<header><div><h1>Screw motion</h1><p>From a target pose to an axis, a rotation, and a translation.</p></div><div class="badge">ATARI LAB · FOUNDATIONS OF MODERN ROBOTICS · L3</div></header>
<main>
<aside class="side"><section class="card"><h2>1 · Define the displacement</h2>
<label class="label" for="preset">Start with an example</label><select id="preset"><option value="helix">Off-axis helix</option><option value="rotation">Pure rotation about an offset axis</option><option value="translation">Pure translation</option><option value="slide33">Slide 33 · find the screw axis</option><option value="halfturn">180° rotation</option><option value="identity">Identity · no motion</option></select>
<label class="label" for="inputMode">Describe the target using</label><select id="inputMode"><option value="pose">Rotation angles + position</option><option value="matrix">Homogeneous transform</option><option value="screw">Screw parameters</option></select>
<div id="posePanel" class="input-panel"><label class="label">Fixed-axis angles (degrees)</label><div class="three"><label>Roll φ<input aria-label="Roll degrees" id="roll" type="number" value="0" step="5"></label><label>Pitch β<input aria-label="Pitch degrees" id="pitch" type="number" value="0" step="5"></label><label>Yaw ψ<input aria-label="Yaw degrees" id="yaw" type="number" value="120" step="5"></label></div><p class="hint">R = R<sub>z</sub>(ψ) R<sub>y</sub>(β) R<sub>x</sub>(φ). The rightmost rotation acts first.</p><label class="label">Target body-origin position p (m)</label><div class="three"><label>x<input aria-label="Target x" id="px" type="number" value="1" step="0.1"></label><label>y<input aria-label="Target y" id="py" type="number" value="0.5" step="0.1"></label><label>z<input aria-label="Target z" id="pz" type="number" value="0.8" step="0.1"></label></div></div>
<div id="matrixPanel" class="input-panel" hidden><label class="label">T₁ = [ R | p ] (last row: 0 0 0 1)</label><div id="matrixFields" class="matrix"></div><p class="hint">Edit the 3 × 4 block, then apply. R must satisfy RᵀR = I and det R = +1.</p></div>
<div id="screwPanel" class="input-panel" hidden><label class="label">Axis direction u (normalized automatically)</label><div class="three"><label>x<input aria-label="Axis direction x" id="ux" type="number" value="0" step="0.1"></label><label>y<input aria-label="Axis direction y" id="uy" type="number" value="0" step="0.1"></label><label>z<input aria-label="Axis direction z" id="uz" type="number" value="1" step="0.1"></label></div><label class="label">A point on the axis (m)</label><div class="three"><label>x<input aria-label="Axis point x" id="qx" type="number" value="0" step="0.1"></label><label>y<input aria-label="Axis point y" id="qy" type="number" value="1" step="0.1"></label><label>z<input aria-label="Axis point z" id="qz" type="number" value="0" step="0.1"></label></div><div class="two"><label class="label">Pitch h (m/rad)<input aria-label="Screw pitch" id="h" type="number" value="0.2" step="0.05"></label><label class="label">Rotation θ (degrees)<input aria-label="Screw angle" id="theta" type="number" value="120" step="15"></label></div><p class="hint">This mode preserves specified turns, including rotations above 180°.</p></div>
<button id="apply" class="primary" style="width:100%;margin-top:12px">Apply displacement</button><div id="status" role="status" aria-live="polite"></div></section>
<section class="card"><h2>2 · Move along the screw</h2><div class="progress"><span>Start: s = 0</span><b id="sLabel">s = 0.40</b><span>Target: s = 1</span></div><input aria-label="Motion progress" id="progress" type="range" min="0" max="1000" step="1" value="400"><div class="buttons"><button id="play" class="primary">▶ Play</button><button id="reset">Reset</button><button id="view">Reset view</button></div><label class="label">Motion duration Δt (s)<input aria-label="Motion duration seconds" id="duration" type="number" min="0.1" max="60" value="3" step="0.5"></label><label class="check"><input id="velocities" type="checkbox" checked> Point-velocity decomposition</label><label class="check"><input id="paths" type="checkbox" checked> Origin and marked-point paths</label><label class="check"><input id="target" type="checkbox" checked> Target body and frame</label><p class="hint">Drag to rotate the view; scroll to zoom. Start pose T₀ = I. All coordinates are in the world frame {s}.</p></section></aside>
<section class="card scene"><div id="plot"></div><div class="legend"><span><i style="background:#7E57C2"></i>Screw axis</span><span><i style="background:#C89800"></i>Marked-point path</span><span><i style="background:#929DA7"></i>Origin path</span><span><i style="background:#DF811C"></i>Rotation velocity</span><span><i style="background:#00989E"></i>Translation velocity</span><span><i style="background:#223349"></i>Total velocity</span><span>Frame axes: <b style="color:#D64045">x</b> / <b style="color:#16865C">y</b> / <b style="color:#2875C7">z</b> (right-handed)</span></div></section>
<div class="readouts"><section class="card"><h2>The screw geometry</h2><div class="equation">T(s) = exp(s ξ<sup>∧</sup>)</div><pre id="geometry"></pre><div class="small" id="branch"></div></section><section class="card"><h2>Connect geometry to twist · slide 32</h2><div class="equation" id="velocityEquation">ẋ = ω × (x − r₀) + hω</div><pre id="velocity"></pre><div class="small">Velocity arrows share a 0.20 s scale. ξ is a pose increment; the physical twist is 𝒱<sub>s</sub> = ξ/Δt. Linear components come first.</div></section><section class="card"><h2>Current pose and body velocity</h2><pre id="poseReadout"></pre><div class="small">𝒱<sub>b</sub> = Ad<sub>T⁻¹</sub> 𝒱<sub>s</sub>. Here T₀ = I and the generator is constant, so 𝒱<sub>b</sub> = 𝒱<sub>s</sub>. This equality is special to this path. In general, v differs from the body-origin velocity ṗ.</div></section></div>
</main><footer>Companion to <i>Robotics from an Optimization Perspective</i>, Majid Khadiv · ATARI Lab, TUM. Original implementation for the course. Teaching reference: <a href="https://github.com/haijunsu-osu/screw_axis" target="_blank" rel="noopener">Screw Axis Visualizer by Haijun Su</a>. Rendering: Plotly.js (MIT). The complete app runs locally in this page; no kernel or network is needed after the file is generated.</footer>
<script>__APP__</script></body></html>
'''

In [ ]:
APP_JS = r'''/* Original course implementation. All twists use translation-first ordering. */
'use strict';
const add=(a,b)=>a.map((x,i)=>x+b[i]), sub=(a,b)=>a.map((x,i)=>x-b[i]);
const scale=(a,s)=>a.map(x=>x*s), dot=(a,b)=>a.reduce((s,x,i)=>s+x*b[i],0);
const norm=a=>Math.hypot(...a), cross=(a,b)=>[a[1]*b[2]-a[2]*b[1],a[2]*b[0]-a[0]*b[2],a[0]*b[1]-a[1]*b[0]];
const eye=()=>[[1,0,0],[0,1,0],[0,0,1]], tr=A=>A[0].map((_,j)=>A.map(r=>r[j]));
const mv=(A,v)=>A.map(r=>dot(r,v)), mm=(A,B)=>A.map(r=>tr(B).map(c=>dot(r,c)));
const madd=(A,B)=>A.map((r,i)=>add(r,B[i])), mscale=(A,s)=>A.map(r=>scale(r,s));
const hat=a=>[[0,-a[2],a[1]],[a[2],0,-a[0]],[-a[1],a[0],0]];
const det=A=>dot(A[0],cross(A[1],A[2]));
function solve3(A,b){
  const C=A.map((r,i)=>[...r,b[i]]);
  for(let j=0;j<3;j++){
    let k=j;for(let i=j+1;i<3;i++)if(Math.abs(C[i][j])>Math.abs(C[k][j]))k=i;
    [C[j],C[k]]=[C[k],C[j]];
    if(Math.abs(C[j][j])<1e-14)throw Error('Singular linear system.');
    const pivot=C[j][j]; C[j]=C[j].map(x=>x/pivot);
    for(let i=0;i<3;i++)if(i!==j){const a=C[i][j];C[i]=C[i].map((x,k)=>x-a*C[j][k]);}
  }
  return C.map(r=>r[3]);
}
function coeff(theta){
  const q=theta*theta;
  return theta<1e-4 ? [1-q/6+q*q/120,.5-q/24+q*q/720,1/6-q/120+q*q/5040]
    : [Math.sin(theta)/theta,2*Math.sin(theta/2)**2/q,(theta-Math.sin(theta))/(q*theta)];
}
function expSE3(xi,s=1){
  const rho=scale(xi.slice(0,3),s), phi=scale(xi.slice(3),s), A=hat(phi), A2=mm(A,A);
  const [a,b,c]=coeff(norm(phi));
  const R=madd(eye(),madd(mscale(A,a),mscale(A2,b)));
  const J=madd(eye(),madd(mscale(A,b),mscale(A2,c)));
  return {R,p:mv(J,rho)};
}
function rotvec(R){
  // Quaternion extraction avoids division by sin(theta) at theta = pi.
  const trace=R[0][0]+R[1][1]+R[2][2];let q;
  if(trace>0){const s=2*Math.sqrt(1+trace);q=[s/4,(R[2][1]-R[1][2])/s,(R[0][2]-R[2][0])/s,(R[1][0]-R[0][1])/s];}
  else{
    let i=0;for(let k=1;k<3;k++)if(R[k][k]>R[i][i])i=k;
    const j=(i+1)%3,k=(i+2)%3,s=2*Math.sqrt(Math.max(0,1+R[i][i]-R[j][j]-R[k][k]));
    q=[(R[k][j]-R[j][k])/s,0,0,0];q[i+1]=s/4;q[j+1]=(R[j][i]+R[i][j])/s;q[k+1]=(R[k][i]+R[i][k])/s;
  }
  q=scale(q,1/norm(q));
  if(q[0]<0)q=scale(q,-1);
  const n=norm(q.slice(1));
  if(n<1e-14)return [0,0,0];
  return scale(q.slice(1),2*Math.atan2(n,q[0])/n);
}
function validPose(T){
  const values=[...T.R.flat(),...T.p];
  if(!values.every(Number.isFinite))throw Error('Enter finite numeric values.');
  const E=sub(mm(tr(T.R),T.R).flat(),eye().flat());
  if(norm(E)>1e-7||Math.abs(det(T.R)-1)>1e-7)throw Error('Invalid R: require RᵀR = I and det R = +1. The last valid scene is retained.');
}
function logSE3(T){
  validPose(T);const phi=rotvec(T.R), A=hat(phi), [,b,c]=coeff(norm(phi));
  const J=madd(eye(),madd(mscale(A,b),mscale(mm(A,A),c)));
  return [...solve3(J,T.p),...phi];
}
function parameters(xi){
  const rho=xi.slice(0,3),phi=xi.slice(3),theta=norm(phi);
  if(theta<1e-12)return {kind:norm(rho)<1e-12?'identity':'translation',theta:0,u:null,r0:null,h:null,d:norm(rho)};
  const u=scale(phi,1/theta),h=dot(phi,rho)/(theta*theta),r0=scale(cross(phi,rho),1/(theta*theta));
  return {kind:Math.abs(h)<1e-12?'rotation':'screw',theta,u,r0,h,d:h*theta};
}
function fromScrew(direction,point,h,theta){
  if(norm(direction)<1e-10)throw Error('Axis direction must be nonzero.');
  const u=scale(direction,1/norm(direction)),r0=sub(point,scale(u,dot(u,point)));
  return [...scale(add(scale(cross(u,r0),-1),scale(u,h)),theta),...scale(u,theta)];
}
function rpy(roll,pitch,yaw){
  const [a,b,c]=[roll,pitch,yaw].map(x=>x*Math.PI/180), ca=Math.cos(a),sa=Math.sin(a),cb=Math.cos(b),sb=Math.sin(b),cc=Math.cos(c),sc=Math.sin(c);
  return [[cc*cb,cc*sb*sa-sc*ca,cc*sb*ca+sc*sa],[sc*cb,sc*sb*sa+cc*ca,sc*sb*ca-cc*sa],[-sb,cb*sa,cb*ca]];
}
function toRPY(R){
  const b=Math.asin(Math.max(-1,Math.min(1,-R[2][0])));let a,c;
  if(Math.abs(Math.cos(b))>1e-8){a=Math.atan2(R[2][1],R[2][2]);c=Math.atan2(R[1][0],R[0][0]);}
  else{a=0;c=Math.atan2(-R[0][1],R[1][1]);}
  return [a,b,c].map(x=>x*180/Math.PI);
}
if(typeof module!=='undefined')module.exports={expSE3,logSE3,parameters,fromScrew,rpy,toRPY,validPose};

if(typeof document!=='undefined'){
const $=id=>document.getElementById(id);
const C={x:'#D64045',y:'#16865C',z:'#2875C7',axis:'#7E57C2',rotation:'#DF811C',translation:'#00989E',total:'#223349'};
let xi=[1,0,1,0,0,2], target=expSE3(xi), source='pose', playing=false, camera=null, busy=false, pending=false;
let valid=true, lastScene=null, animationStart=0, animationS=0;
const pointB=[.32,.22,.17], arrowScale=.2;
const number=id=>{const v=Number($(id).value);if($(id).value.trim()===''||!Number.isFinite(v)||Math.abs(v)>1e5)throw Error('Enter finite numbers with magnitude below 100000.');return v;};
const vec=ids=>ids.map(number);
const fmt=x=>Math.abs(x)<.0005?'0.000':x.toFixed(3), vector=v=>'['+v.map(fmt).join(', ')+']';
const setNumber=(id,value)=>$(id).value=Number(value.toPrecision(12));
for(let i=0;i<3;i++)for(let j=0;j<4;j++){
  const input=document.createElement('input');input.type='number';input.step='0.1';input.id=`m${i}${j}`;
  input.setAttribute('aria-label',`Transform row ${i+1} column ${j+1}`);$('matrixFields').append(input);
}
function syncFields(){
  toRPY(target.R).forEach((v,i)=>setNumber(['roll','pitch','yaw'][i],v));
  target.p.forEach((v,i)=>setNumber(['px','py','pz'][i],v));
  for(let i=0;i<3;i++)for(let j=0;j<4;j++)setNumber(`m${i}${j}`,j<3?target.R[i][j]:target.p[i]);
  const p=parameters(xi);
  if(p.u){p.u.forEach((v,i)=>setNumber(['ux','uy','uz'][i],v));p.r0.forEach((v,i)=>setNumber(['qx','qy','qz'][i],v));setNumber('h',p.h);setNumber('theta',p.theta*180/Math.PI);}
}
function stop(){playing=false;$('play').textContent='▶ Play';}
function status(message,error=false){$('status').textContent=message;$('status').className=error?'error':'';}
function apply(){
  stop();try{
    const mode=$('inputMode').value;let next;
    if(mode==='pose')next=logSE3({R:rpy(...vec(['roll','pitch','yaw'])),p:vec(['px','py','pz'])});
    if(mode==='matrix')next=logSE3({R:[0,1,2].map(i=>[0,1,2].map(j=>number(`m${i}${j}`))),p:[0,1,2].map(i=>number(`m${i}3`))});
    if(mode==='screw')next=fromScrew(vec(['ux','uy','uz']),vec(['qx','qy','qz']),number('h'),number('theta')*Math.PI/180);
    const duration=number('duration');if(duration<.1||duration>60)throw Error('Motion duration must be between 0.1 and 60 seconds.');
    xi=next;target=expSE3(xi);source=mode;valid=true;syncFields();status('Displacement applied.');requestRender();
  }catch(e){valid=false;status(e.message,true);}
}
function changeMode(){
  const mode=$('inputMode').value;
  ['pose','matrix','screw'].forEach(m=>$(m+'Panel').hidden=m!==mode);
  status(mode==='screw'?'Edit axis, pitch, or angle, then apply.':'Edit the target, then apply.');
}
function preset(name){
  stop();let next,duration=3;
  if(name==='helix')next=fromScrew([0,0,1],[0,1,0],.2,2*Math.PI/3);
  if(name==='rotation')next=fromScrew([0,0,1],[0,1,0],0,2*Math.PI/3);
  if(name==='translation')next=[1,.5,1,0,0,0];
  if(name==='slide33'){duration=.5;next=scale([4,0,0,0,2,2],duration);}
  if(name==='halfturn')next=fromScrew([1,1,0],[0,0,0],.15,Math.PI);
  if(name==='identity')next=[0,0,0,0,0,0];
  xi=next;target=expSE3(xi);source='pose';valid=true;setNumber('duration',duration);
  $('inputMode').value='pose';$('progress').value=400;changeMode();syncFields();
  status(name==='slide33'?'𝒱s = (4, 0, 0, 0, 2, 2); duration = 0.5 s.':'Displacement applied.');requestRender();
}
function line(points,color,label,width=4,dash='solid',opacity=1,labels=null){
  return {type:'scatter3d',x:points.map(p=>p?p[0]:null),y:points.map(p=>p?p[1]:null),z:points.map(p=>p?p[2]:null),
    mode:labels?'lines+text':'lines',text:labels,textposition:'top center',textfont:{color,size:12},
    line:{color,width,dash},name:label,hoverinfo:'name',showlegend:false,opacity};
}
function arrow(start,v,color,label,opacity=1,showText=true){
  const n=norm(v);if(n<1e-10)return line([],color,label);
  const d=scale(v,1/n),tip=add(start,v),a=Math.abs(d[2])<.85?[0,0,1]:[0,1,0];
  const b=scale(cross(d,a),1/norm(cross(d,a))),len=Math.min(.11,n*.23),base=sub(tip,scale(d,len));
  return line([start,tip,null,add(base,scale(b,len*.45)),tip,sub(base,scale(b,len*.45))],color,label,5,'solid',opacity,showText?['',label,'','','','']:null);
}
function frame(T,label,opacity=1){
  return ['x','y','z'].map((axis,j)=>arrow(T.p,scale(T.R.map(r=>r[j]),.5),C[axis],`${axis}_${label}`,opacity,label==='b'));
}
function body(T,color,opacity=1){
  const verts=[];for(const x of [-.32,.32])for(const y of [-.22,.22])for(const z of [-.17,.17])verts.push([x,y,z]);
  const world=verts.map(p=>add(mv(T.R,p),T.p)),edges=[];
  for(let i=0;i<8;i++)for(let j=i+1;j<8;j++)if(verts[i].filter((v,k)=>v!==verts[j][k]).length===1)edges.push(world[i],world[j],null);
  return line(edges,color,'Rigid body',4,'solid',opacity);
}
function makeScene(s){
  const P=parameters(xi),T=expSE3(xi,s),duration=number('duration');
  if(duration<.1||duration>60)throw Error('Motion duration must be between 0.1 and 60 seconds.');
  const V=scale(xi,1/duration),v=V.slice(0,3),w=V.slice(3),x=add(mv(T.R,pointB),T.p);
  const total=add(v,cross(w,x)),rot=P.u?cross(w,sub(x,P.r0)):[0,0,0],translation=P.u?scale(w,P.h):v;
  const origins=[],marked=[];for(let i=0;i<=100;i++){const pose=expSE3(xi,i/100);origins.push(pose.p);marked.push(add(mv(pose.R,pointB),pose.p));}
  const traces=[...frame({R:eye(),p:[0,0,0]},'s',.7),...frame(T,'b'),body(T,'#506A82')];
  const bounds=[...origins,...marked,[0,0,0],[.6,.6,.6]];
  if($('target').checked)traces.push(...frame(target,'1',.35),body(target,'#9AA8B8',.45));
  if(P.u){
    const extent=Math.max(.65,Math.abs(P.d)+.35),axis=[add(P.r0,scale(P.u,-extent)),add(P.r0,scale(P.u,extent))];
    traces.push(line(axis,C.axis,'Screw axis',5,'dash'),arrow(P.r0,scale(P.u,.7),C.axis,'u'),
      line([[0,0,0],P.r0],'#B3A5D1','r₀ ⟂ u',2,'dot'));
    const foot=add(P.r0,scale(P.u,dot(P.u,sub(T.p,P.r0))));
    traces.push(line([T.p,foot],'#8A8A8A','Radius to screw axis',2,'dot'),
      {type:'scatter3d',x:[P.r0[0]],y:[P.r0[1]],z:[P.r0[2]],mode:'markers+text',text:['r₀'],textposition:'bottom left',marker:{size:4,color:C.axis},showlegend:false});
    bounds.push(...axis);
  }
  if($('paths').checked)traces.push(line(origins,'#929DA7','Body-origin path',4),line(marked,'#C89800','Marked-point path',4));
  traces.push({type:'scatter3d',x:[x[0]],y:[x[1]],z:[x[2]],mode:'markers+text',text:['x'],textposition:'top right',marker:{size:5,color:'#C89800'},showlegend:false,name:'Marked body point'});
  if($('velocities').checked){
    traces.push(arrow(x,scale(rot,arrowScale),C.rotation,'ω × (x − r₀)',1,false),
      arrow(add(x,scale(rot,arrowScale)),scale(translation,arrowScale),C.translation,P.u?'hω':'v',1,false),
      arrow(x,scale(total,arrowScale),C.total,'ẋ',1,false));
    // Bound velocity arrows for the whole path to prevent the axes rescaling during play.
    for(const point of marked){const rv=P.u?cross(w,sub(point,P.r0)):[0,0,0];bounds.push(add(point,scale(rv,arrowScale)),add(point,scale(add(v,cross(w,point)),arrowScale)));}
  }
  const ranges=[0,1,2].map(j=>[Math.min(...bounds.map(p=>p[j]))-.85,Math.max(...bounds.map(p=>p[j]))+.85]);
  const layout={margin:{l:0,r:0,b:0,t:8},paper_bgcolor:'white',font:{family:'Arial',size:12},uirevision:'screw-camera',
    scene:{aspectmode:'data',xaxis:{title:{text:'x_s (m)'},range:ranges[0]},yaxis:{title:{text:'y_s (m)'},range:ranges[1]},zaxis:{title:{text:'z_s (m)'},range:ranges[2]},
    camera:camera||{eye:{x:1.25,y:-1.6,z:1.05}},dragmode:'orbit'}};
  const pdot=add(v,cross(w,T.p)),Vb=[...mv(tr(T.R),pdot),...mv(tr(T.R),w)];
  return {P,T,V,Vb,x,rot,translation,total,pdot,traces,layout,s,duration};
}
function readouts(S){
  const {P,T,V,Vb,rot,translation,total,pdot,s,duration}=S;
  $('sLabel').textContent=`s = ${s.toFixed(2)}`;
  $('velocityEquation').textContent=P.u?'ẋ = ω × (x − r₀) + hω':P.kind==='translation'?'ω = 0: ẋ = v':'No motion: ẋ = 0';
  $('geometry').textContent=P.u?`u  = ${vector(P.u)}\nr₀ = ${vector(P.r0)} m\nθ  = ${fmt(P.theta)} rad (${fmt(P.theta*180/Math.PI)}°)\nh  = ${fmt(P.h)} m/rad\nAxial slide d = hθ = ${fmt(P.d)} m\nAxis: r(λ) = r₀ + λu`:
    P.kind==='translation'?`Pure translation\np₁ = ${vector(target.p)} m\nDistance = ${fmt(norm(target.p))} m\nNo unique finite screw axis or finite pitch.`:
    'Identity displacement\nNo motion; axis and pitch are undetermined.';
  $('branch').textContent=source==='screw'?'The specified angle and winding are preserved. At θ = 0 this finite-pitch screw gives the identity.':
    `Pose input uses a principal rotation angle in [0, π]. Endpoints alone cannot determine extra turns.${Math.abs(P.theta-Math.PI)<1e-6?' At π, either axis sign gives a valid branch; this app chooses one.':''}`;
  $('velocity').textContent=`𝒱s = (v, ω)\nv = ${vector(V.slice(0,3))} m/s\nω = ${vector(V.slice(3))} rad/s\n\nRotation:    ${vector(rot)} m/s\nTranslation: ${vector(translation)} m/s\nTotal ẋ:     ${vector(total)} m/s`;
  const rows=T.R.map((r,i)=>'[ '+[...r,T.p[i]].map(x=>fmt(x).padStart(7)).join(' ')+' ]');
  $('poseReadout').textContent=`T_sb(s) =\n${rows.join('\n')}\n[   0.000   0.000   0.000   1.000 ]\n\n𝒱b = ${vector(Vb)}\nṗ  = ${vector(pdot)} m/s\nTime: ${fmt(s*duration)} / ${fmt(duration)} s`;
}
async function requestRender(){
  if(busy){pending=true;return;}busy=true;
  try{const S=makeScene(Number($('progress').value)/1000);lastScene=S;readouts(S);
    await Plotly.react('plot',S.traces,S.layout,{responsive:true,displaylogo:false,scrollZoom:true});
  }catch(e){status(e.message,true);stop();}finally{busy=false;if(pending){pending=false;requestRender();}}
}
function tick(now){
  if(!playing)return;
  const s=Math.min(1,animationS+(now-animationStart)/(1000*number('duration')));
  $('progress').value=Math.round(s*1000);requestRender();
  if(s>=1)stop();else requestAnimationFrame(tick);
}
$('apply').addEventListener('click',apply);$('inputMode').addEventListener('change',changeMode);
$('preset').addEventListener('change',e=>preset(e.target.value));
$('progress').addEventListener('input',()=>{stop();requestRender();});
$('play').addEventListener('click',()=>{
  if(playing){stop();return;}if(!valid){status('Fix the invalid input and apply the displacement before playing.',true);return;}
  try{const d=number('duration');if(d<.1||d>60)throw Error('Motion duration must be between 0.1 and 60 seconds.');}
  catch(e){status(e.message,true);return;}
  animationS=Number($('progress').value)/1000;if(animationS>=1){animationS=0;$('progress').value=0;}
  playing=true;animationStart=performance.now();$('play').textContent='Ⅱ Pause';requestAnimationFrame(tick);
});
$('reset').addEventListener('click',()=>{stop();$('progress').value=0;requestRender();});
$('view').addEventListener('click',()=>{camera=null;Plotly.relayout('plot',{'scene.camera':{eye:{x:1.25,y:-1.6,z:1.05}}});});
['velocities','paths','target'].forEach(id=>$(id).addEventListener('change',requestRender));
$('duration').addEventListener('change',()=>{stop();requestRender();});
['posePanel','screwPanel'].forEach(id=>$(id).addEventListener('change',apply));
// Keep a user-selected camera while fields and time are changed.
preset('helix');
setTimeout(()=>{if($('plot').on)$('plot').on('plotly_relayout',e=>{if(e['scene.camera'])camera=e['scene.camera'];});},300);
}
'''

In [ ]:
from pathlib import Path
from html import escape
from plotly.offline import get_plotlyjs
from IPython.display import HTML, FileLink, display

gui_html = HTML_TEMPLATE.replace('__PLOTLY__', get_plotlyjs()).replace('__APP__', APP_JS)
output_file = Path('L3_screw_motion.html')
output_file.write_text(gui_html, encoding='utf-8')
display(FileLink(str(output_file), result_html_prefix='Open the full browser GUI: '))
display(HTML('<iframe title="L3 screw motion GUI" style="width:100%;height:1120px;border:1px solid #ddd;border-radius:8px" '
             'srcdoc="' + escape(gui_html, quote=True) + '"></iframe>'))


## References and implementation notes

- Course: L3, slides 32–33, and Chapter 3, *Rigid-Body Motion*.
- Teaching reference: [Haijun Su's Screw Axis Visualizer](https://github.com/haijunsu-osu/screw_axis). This is a new implementation using the course notation; no source code or assets were copied from that project.
- [Solà, Deray, and Atchuthan, *A micro Lie theory for state estimation in robotics*](https://arxiv.org/abs/1812.01537).
- [Plotly.js documentation](https://plotly.com/javascript/3d-line-plots/); the generated HTML embeds Plotly.js with its existing MIT license header.

The numerical rotation checks accept errors up to $10^{-7}$ in browser matrix inputs. Rotations below $10^{-12}$ radians are treated as translation for axis reporting. The full exponential still uses a Taylor expansion near zero. Very small nonzero rotations can imply a very distant screw axis: this is a geometric conditioning issue, not a large physical displacement.
